In [ ]:
# ── CELL 1 — Install correct versions for PyTorch 2.10 + CUDA 12.8 ───────────
!pip install -q \
    numpy==1.26.4 \
    transformers==4.44.0 \
    trl==0.9.6 \
    peft==0.12.0 \
    bitsandbytes==0.45.5 \
    accelerate==0.34.0 \
    datasets==2.21.0

print(" Packages installed")

In [ ]:
import torch

try:
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        vram     = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f' GPU found: {gpu_name}')
        print(f' VRAM: {vram:.1f} GB')
    else:
        print(' No GPU — go to Runtime → Change runtime type → T4 GPU')
except Exception as e:
    print(f' Error: {e}')

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
os.makedirs('data',     exist_ok=True)
os.makedirs('adapters', exist_ok=True)
os.makedirs('/content/drive/MyDrive/Week8/adapters', exist_ok=True)
print(' Folders ready')

In [ ]:
import shutil

try:
    shutil.copy('/content/drive/MyDrive/Week8/data/train.jsonl', 'data/train.jsonl')
    shutil.copy('/content/drive/MyDrive/Week8/data/val.jsonl',   'data/val.jsonl')
    print(' Dataset copied from Google Drive')
except Exception as e:
    print(f' Error: {e}')
    print('Make sure Day 1 was completed and files saved to Drive')

In [ ]:
import json

def load_jsonl(filepath):
    samples = []
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip(): samples.append(json.loads(line.strip()))
    except Exception as e: print(f'Error: {e}')
    return samples

train_data = load_jsonl('data/train.jsonl')
val_data   = load_jsonl('data/val.jsonl')

print(f' Train: {len(train_data)} | Val: {len(val_data)}')
print(f'   instruction: {train_data[0]["instruction"]}')
print(f'   input      : {train_data[0]["input"][:80]}...')
print(f'   output     : {train_data[0]["output"][:80]}...')

In [ ]:
# converts each sample into TinyLlama chat format:
#   <|system|>  You are a helpful medical assistant.
#   <|user|>    {instruction}\n{input}
#   <|assistant|> {output}

def format_prompt(sample):
    try:
        instruction = sample['instruction']
        inp         = sample['input']
        output      = sample['output']
        user_msg    = f'{instruction}\n{inp}' if inp else instruction
        return (
            f'<|system|>\nYou are a helpful medical assistant.\n'
            f'<|user|>\n{user_msg}\n'
            f'<|assistant|>\n{output}'
        )
    except Exception as e:
        print(f' Error: {e}'); return None

# test on one sample
print(format_prompt(train_data[0]))

In [ ]:
from datasets import Dataset

def convert_to_hf_dataset(data):
    try:
        formatted = [{'text': format_prompt(s)} for s in data if format_prompt(s)]
        return Dataset.from_list(formatted)
    except Exception as e:
        print(f' Error: {e}'); return None

train_dataset = convert_to_hf_dataset(train_data)
val_dataset   = convert_to_hf_dataset(val_data)
print(f'Train HF dataset: {len(train_dataset)} | Val: {len(val_dataset)}')

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

try:
    print('⏳ Loading tokenizer...')
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.padding_side = 'right'
    print(f' Tokenizer loaded | Vocab size: {tokenizer.vocab_size}')
except Exception as e:
    print(f'Error: {e}')

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# loads the model using only 4GB VRAM instead of 8GB
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

try:
    print('Loading TinyLlama in 4-bit... (1-2 mins)')
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map='auto'
    )
    model.config.use_cache = False
    vram_used = torch.cuda.memory_allocated() / 1024**3
    print(f' Model loaded | VRAM used: {vram_used:.1f} GB')
except Exception as e:
    print(f' Error: {e}')

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

try:
    model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=16,                  # rank
        lora_alpha=32,          # scaling (2x rank)
        lora_dropout=0.05,
        bias='none',
        task_type='CAUSAL_LM',
        target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj']
    )
    model = get_peft_model(model, lora_config)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f'LoRA attached | Trainable: {trainable:,} ({100*trainable/total:.2f}%)')
    print(f'   Total params : {total:,} | Frozen: {total-trainable:,}')
except Exception as e:
    print(f' Error: {e}')

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='adapters',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,  # effective batch = 16
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    eval_strategy='steps',
    eval_steps=20,
    save_steps=20,
    save_total_limit=2,
    load_best_model_at_end=True,
    warmup_steps=10,
    report_to='none',
    gradient_checkpointing=True,
)
print(' Training args set: epochs=3, lr=2e-4, batch=4, fp16=True')

In [ ]:
from trl import SFTTrainer

try:
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        dataset_text_field='text',
        max_seq_length=512,
        packing=False,
    )
    print('Training started... watch the loss go down!')
    trainer.train()
    print(' Training complete!')
except Exception as e:
    print(f' Error: {e}')

In [ ]:
try:
    model.save_pretrained('adapters')
    tokenizer.save_pretrained('adapters')
    print(' Saved to adapters/')

    model.save_pretrained('/content/drive/MyDrive/Week8/adapters')
    tokenizer.save_pretrained('/content/drive/MyDrive/Week8/adapters')
    print(' Saved to Google Drive')

    print(f'   Files: {os.listdir("adapters")}')
except Exception as e:
    print(f' Error: {e}')

In [ ]:
def generate_response(question, max_new_tokens=150):
    try:
        prompt = (
            f'<|system|>\nYou are a helpful medical assistant.\n'
            f'<|user|>\n{question}\n'
            f'<|assistant|>\n'
        )
        inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.7,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )
        return tokenizer.decode(
            outputs[0][inputs['input_ids'].shape[1]:],
            skip_special_tokens=True
        )
    except Exception as e:
        return f' Error: {e}'

question = 'What are the common symptoms of diabetes?'
print(f'Q: {question}')
print(f'A: {generate_response(question)}')

In [ ]:
import matplotlib.pyplot as plt

try:
    log_history  = trainer.state.log_history
    train_steps  = [x['step'] for x in log_history if 'loss' in x and 'eval_loss' not in x]
    train_losses = [x['loss'] for x in log_history if 'loss' in x and 'eval_loss' not in x]
    eval_steps   = [x['step'] for x in log_history if 'eval_loss' in x]
    eval_losses  = [x['eval_loss'] for x in log_history if 'eval_loss' in x]

    plt.figure(figsize=(10, 4))
    plt.plot(train_steps, train_losses, label='Train Loss', color='steelblue')
    plt.plot(eval_steps,  eval_losses,  label='Val Loss',   color='coral')
    plt.title('Training Loss over Steps')
    plt.xlabel('Steps'); plt.ylabel('Loss')
    plt.legend(); plt.tight_layout()
    plt.savefig('adapters/loss_curve.png'); plt.show()
    print(' Loss curve saved')
except Exception as e:
    print(f' Error: {e}')

In [ ]:
import os
print('=' * 40 + '\nDAY 2 — COMPLETION CHECK\n' + '=' * 40)

files = [
    ('adapters/adapter_model.safetensors', 'LoRA weights'),
    ('adapters/adapter_config.json',       'LoRA config'),
    ('adapters/tokenizer.json',            'Tokenizer'),
    ('adapters/loss_curve.png',            'Loss curve'),
]

# some versions save as .bin instead of .safetensors
if not os.path.exists('adapters/adapter_model.safetensors'):
    files[0] = ('adapters/adapter_model.bin', 'LoRA weights')

all_good = True
for fp, desc in files:
    if os.path.exists(fp):
        size = os.path.getsize(fp) / 1024**2
        print(f' {desc}: {fp} ({size:.1f} MB)')
    else:
        print(f' MISSING: {fp}'); all_good = False

print('\n Day 2 Complete! Ready for Day 3 (Quantisation)' if all_good else '\n Re-run cells above.')